# Root Cause Analysis — Myntra Case Study

**Course:** SST Data Analytics 101 · Term 4 · Batch 2029  
**Topic:** RCA framework + clickstream funnel diagnosis  


---
## 1 · What is Root Cause Analysis?

### What is a Root Cause?

A **root cause** is the core issue — the **underlying cause** — of a problem.

### What is Root Cause Analysis (RCA)?

- A **systematic process** for identifying the underlying causes of a problem or issue.
- A collective term for a wide range of **approaches, tools, and techniques** used to uncover causes of problems.
- Involves **breaking down** the problem into smaller parts, examining each part in detail, and identifying the root causes contributing to the problem.

#### Everyday examples

| Situation | Who finds the root cause? |
|-----------|---------------------------|
| Sick and throwing up at work | Doctor |
| Car stops working | Mechanic |
| Business underperforming in an area | Analyst / product team |

### Goals of RCA

1. **Discover** the root cause of a problem or event.
2. **Understand** how to fix, compensate, or learn from underlying issues.
3. **Apply** what you learn to systematically prevent future issues — or repeat successes.

### Three fundamental questions

Within an organization, problem solving, incident investigation, and RCA are connected by:

| Question | What to do |
|----------|------------|
| **What's the problem?** | Define the issue by its impact on overall goals. |
| **Why did it happen?** | List potential causal factors → determine the root cause. |
| **What will prevent it?** | Select the best solution to prevent negative impact. |

---
## 2 · Life Cycle of RCA

General guidelines / steps to carry out an analysis:

### Clarify
- Ask questions to get enough clarity on the problem.
- Clarify terms and set up parameters for discussion.
- Create an outline of the approach you're going to follow.

### Rule Out
- Explore possibilities and list high-level causes.
- Discard issues that seem out of scope.
- Check underlying data for accuracy.
- Rule out technical glitches, bugs, or outliers.
- List observations and start diagnosing.

### Internal Factors
- Recent features / products launched.
- Recent changes made by other teams.
- Slice & dice data by user demographics.

### External Factors
- Bad PR or controversy related to the company.
- Changes in user behavior or customer trends.
- Macroeconomic or geographical changes.
- Competitor analysis.

Based on the analysis you will get a point of issue to investigate further. Then reach a conclusion and suggest a fix.

> **Note:** Capture and report **all** findings — no matter how small they seem.

---
## 3 · Myntra Case Study — Problem Statement

**Myntra has observed a decline from 5% to 3% (i.e., −2 percentage points) in confirmed orders over the last 3 days.**

Your task: perform a Root Cause Analysis and help diagnose the issue.

### Clickstream Data

Clickstream data is information collected about a user while they browse a website or app:

- Unique vs repeat visitor
- Search terms used
- Landing page
- Time spent on each page
- Features clicked / engaged with
- When items are added or removed from cart
- Navigation path (where the user goes next)
- Back-button usage

A **single session** may not be very useful on its own. **Aggregated** clickstream data from many visitors helps improve the product.

**Examples:**
- Many users leave after a page with too little info → enrich that page.
- Users often land on a non-homepage page → redesign it to be more inviting.

### Benefits of clickstream analysis

| Benefit | What you learn |
|---------|----------------|
| **User information** | Search terms, pages visited, cart actions |
| **User routes** | Paths users take to reach a page or purchase |
| **Customer trends** | Traffic sources, time on page, page visits, unique vs repeat |
| **UX** | Pages where users drop off quickly → optimization needed |
| **Digital marketing** | Which ads / campaigns drive conversion; best times to run campaigns |

---
## 4 · Establishing Assumptions

Before diving into data, **align on definitions** with stakeholders.

### Q. How do we define cart additions?

- A user clicks an item on the product page and adds it to their **cart** (shopping list).
- They can later view, add, or remove items from the cart.

### Q. How are we defining conversion?

At Myntra, the metric is **orders per session**:

$$\text{Orders per session} = \frac{\text{No. of orders placed in a day}}{\text{Total no. of sessions that day}}$$

One order can contain multiple items.

### Q. How do we define a session?

- Browsing period between **login** and **logout**.
- One session can have multiple orders or cart additions.

### Q. What is a bounce session?

- User lands on **only one page** (Homepage) and leaves.

$$\text{Bounce rate} = \frac{\text{No. of bounce sessions in a day}}{\text{Total no. of sessions that day}}$$

---
## 5 · User Journey Funnel

### Basic flowchart

![Myntra user journey funnel](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/057/132/original/PA7.jpeg?1700231506)

### Visualizing customer journey — Sankey diagram

*For instructor reference:*

![Sankey diagram — customer journey](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/057/133/original/PA8.png?1700231529)

---
## 6 · Estimation & Backtracking

### Creating a rough estimate (baseline funnel)

Run the cell below to walk through the **5% conversion** baseline before the decline.

In [ ]:
import pandas as pd

# Baseline assumptions (before decline)
sessions = 100
bounce_rate = 0.10
non_bounced = sessions * (1 - bounce_rate)
product_page = round(non_bounced * (1/3))          # ~1/3 of non-bounced sessions
add_to_cart = round(product_page * 0.50)           # 50% of product viewers
checkout = 12                                      # given
payments = 6                                       # given
confirmed_orders = 5                               # given
conversion_rate = confirmed_orders / sessions

funnel = pd.DataFrame({
    "Stage": [
        "Total sessions",
        "Non-bounced sessions (90%)",
        "Product page view (~33% of non-bounced)",
        "Add to cart / Buy now (~50% of product views)",
        "Checkout",
        "Payments",
        "Confirmed orders",
    ],
    "Count": [
        sessions,
        int(non_bounced),
        product_page,
        add_to_cart,
        checkout,
        payments,
        confirmed_orders,
    ],
})

funnel["Step conversion"] = funnel["Count"].pct_change().fillna(1).round(3) * 100
print(f"Baseline conversion rate (orders / sessions): {conversion_rate:.1%}")
display(funnel)

Baseline conversion rate (orders / sessions): 5.0%


,Stage,Count,Step conversion
0,Total sessions,100,100.0
1,Non-bounced sessions (90%),90,-10.0
2,Product page view (~33% of non-bounced),30,-66.7
3,Add to cart / Buy now (~50% of product views),15,-50.0
4,Checkout,12,-20.0
5,Payments,6,-50.0
6,Confirmed orders,5,-16.7


> **Necessary assumptions are made** in the numbers above — treat them as a teaching scaffold, not audited production data.

### Backtracking the user journey

**Since confirmed orders are down by 2 pp (5% → 3%), possible causes include:**

- Drop in total user sessions per day X
- Increase in bounce rate X
- Users not proceeding past the product page (product dislike) X
- Users unable to add items to cart
- Issues during checkout or payment

### Observations (stakeholder data)

| Check | Finding |
|-------|---------|
| Overall user activity | **No decline** — traffic is normal |
| Bounce rate | **Stagnant at ~10%** |
| Product page — avg rating | 3.8–4.2 across categories (unchanged) |
| Expected delivery | 5–7 days (unchanged) |
| Discounts | 20–45% across categories (unchanged) |

**Conclusion so far:** The issue likely lies in the **last 3 stages** of the funnel:

1. Add to Cart  
2. Checkout  
3. Make Payment

---
## 7 · Ask Clarifying Questions

Use the **Clarify → Rule Out → Internal / External** framework. Below are sample Q&A with stakeholders.

### External factors

**Q:** Competitors like Ajio — any upcoming sale or campaign?

**A:** Limited knowledge. Social media doesn't show anything obvious, but hidden targeting is possible.

### Demographic details

**Q:** Is the decline concentrated in a particular region or city?

**A:** No — spread across many locations.

**Q:** Any specific gender or age group affected?

**A:** No — uniformly distributed across groups.

### Macro-economic / supply chain

**Q:** Recent partnership changes? Merchant pull-outs? Stock issues?

**A:** No such events. Majority of products are in stock.

### Product / app changes

**Q:** Any recent changes or upgrades to the application?

**A:** **Yes** — certain upgrades were deployed in the last few days.

---

**Discussion prompt:** Given the funnel backtracking and these answers, where would you investigate next?

---
## 8 · Diagnose the Root Cause

### Q1. Changes to "Add to cart" or Checkout?

**A:** Due to a version upgrade, the **complete shopping flow** changed. Checkout was improved for a seamless experience. However, the **placement and design of "Add to cart"** was left untouched.

### Q2. Bug reports for the new update?

**A:** Not many yet — the update launched only a few days ago.

### Q3. Missing leading payment merchants?

**A:** No — Paytm, PhonePe, GPay, etc. are all integrated.

### Q4. Are payment gateways working properly?

**A:** **Yes — complaints received.** Some customers faced issues while trying to pay during checkout.

---

**Discussion prompt:** Cart additions still happen, but confirmed orders drop. What hypothesis does that support?

---
## 9 · Payment-Related Issues

Let's explore payment issues in detail.

![Payment gateway reliability comparison](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/057/134/original/PA9.jpeg?1700231548)

> **Note:** Data fetched from a third-party application that compares reliability of different payment gateways.

---
## 10 · Conclusions

1. **3 out of 5** partnering banking services providing payment options on Myntra are experiencing **frequent issues and high server downtime** in the current week.
2. This **validates customer complaints** about incomplete transactions and payment failures.
3. If payment fails multiple times, customers **will not proceed** — risk of losing money.
4. Result: **automatic decrease** in confirmed orders over the week.
5. People are **still adding items to carts** → no critical issue with the app UX itself; recovery expected once server issues are resolved.

### RCA summary (fill in class)

| Step | Myntra case |
|------|-------------|
| **Problem** | Confirmed-order rate dropped 5% → 3% over 3 days |
| **Why** | Payment gateway downtime (3/5 partners) after app upgrade week |
| **Prevention** | Monitor gateway SLAs, failover routing, alert on payment-failure spike |

---

### Optional wrap-up questions

1. Why is it important to **define metrics** (session, conversion, bounce) before analyzing?
2. How does **funnel backtracking** narrow the search space?
3. What **external vs internal** factors did we rule out, and how?
4. What would you recommend Myntra do **this week** vs **long term**?

---
*Instructor note: Allow 5–10 min break before Section 7 if running a long session. Source: [HackMD — PA Lec-4](https://hackmd.io/ukaFk_jvRVOO2EEmnP1WMA)*